# Training an anomaly detector

This notebook does the same as `train.py`, step by step. Once you're happy with a setup, use the script
to run longer trainings (see `run-training.sh`).

<!-- TODO(organisers): explain the autoencoder idea: learn to reconstruct normal events, and flag events that reconstruct badly. -->

In [ ]:
import numpy as np
from tensorflow import keras

from dataloader import create_datasets
from models.autoencoder import make_target, masked_mse
from models.registry import REGISTRY
from utils import count_parameters
import plotting

In [ ]:
data_config = "config/baseline.yml"
model_type = "dense-ae"

print("Available models:", list(REGISTRY))
model_cfg = REGISTRY[model_type]
input_type = model_cfg["input_type"]

train, val, _ = create_datasets(data_config, input_type=input_type)
print(f"{len(train):,} training events, {len(val):,} validation events")

In [ ]:
# The model inputs, after the scaling defined in the config
plotting.plot_data_distributions(train);

In [ ]:
input_shape = train.x.shape[1:]
model = model_cfg["build"](input_shape)

model.summary()
print(f"Number of trainable parameters: {count_parameters(model)}")

In [ ]:
# The loss is the reconstruction error of each event, ignoring padded objects
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=masked_mse)

In [ ]:
N_EPOCHS = 5

fit = model.fit(
    train.x, make_target(train.x, train.mask),  # the target also carries the mask, for the loss
    validation_data=(val.x, make_target(val.x, val.mask)),
    epochs=N_EPOCHS,
    batch_size=1024,
    shuffle=True,
)

epochs = range(1, len(fit.history["loss"]) + 1)
history = {"train": list(zip(epochs, fit.history["loss"])), "val": list(zip(epochs, fit.history["val_loss"]))}
plotting.plot_loss_curve(history);

In [ ]:
from pathlib import Path

OUT_PATH = "output"
Path(OUT_PATH, model_type).mkdir(parents=True, exist_ok=True)
model.save(f"{OUT_PATH}/{model_type}/best_model.h5")

## Next steps

<!-- TODO(organisers): what to try next, e.g. other models in models/, different configs, adding your own model. -->

Evaluate your model in [`evaluate.ipynb`](evaluate.ipynb).